# Image Cluster Notebook

In [1]:
# PROJ must be configured before rasterio/localtileserver are imported.
import os
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"

from pathlib import Path

import ipysheet
from IPython.display import Markdown, display
import ipywidgets
import leafmap
import numpy
import pandas
import rasterio
from rasterio.windows import Window
from localtileserver import TileClient, get_leaflet_tile_layer
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from ipyleaflet import WidgetControl

from ImageCluster.model.Clusterer import Clusterer
from ImageCluster.model.ImageHelperSingle import ImageHelper

# Configuration

In [2]:
# Original full-resolution lunar raster.
inFile = Path(
    "/explore/nobackup/projects/lfm/processed_data/Lunar/"
    "LRO_NAC_Pho_Sites/M1126986303LE.ech.cog.tif"
)

NAC_NODATA = -3.40282265508890445e+38
noDataValue = NAC_NODATA
numClusters = 60

# Explicit working crop size. Clustering will ONLY run on this clipped raster.
clipSize = 512

# Outputs are written here.
outDirectory = Path(".")
outDirectory.mkdir(parents=True, exist_ok=True)

clippedInputFile = outDirectory / (
    f"{inFile.stem}-clip-{clipSize}{inFile.suffix}"
)
labelsFile = outDirectory / (
    f"{inFile.stem}-clip-{clipSize}-labels{inFile.suffix}"
)
clusterMapFile = outDirectory / (
    f"{inFile.stem}-clip-{clipSize}-cluster-map{inFile.suffix}"
)

# Helper functions

In [3]:
def crop_center(src_path: Path, dst_path: Path, size: int = 512) -> Path:
    """Write a centered square crop while preserving CRS/georeferencing."""
    with rasterio.open(src_path) as src:
        if src.width < size or src.height < size:
            raise ValueError(
                f"Raster is only {src.width}x{src.height}; "
                f"cannot make a {size}x{size} crop."
            )

        col_off = (src.width - size) // 2
        row_off = (src.height - size) // 2
        window = Window(col_off, row_off, size, size)

        data = src.read(window=window)
        transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.update(
            width=size,
            height=size,
            transform=transform,
        )

        with rasterio.open(dst_path, "w", **profile) as dst:
            dst.write(data)

    return dst_path


# ----------------------------------------------------------------------------
# handleClick
# ----------------------------------------------------------------------------
def handleClick(change: dict) -> None:
    with output:
        if change.new == "Next":
            if not sl.value:
                print("Select at least one cluster before clicking Next.")
                bt.value = "Select:"
                return

            nn = updateList(list(sl.options), list(sl.value))
            updateDict("N")
            sl.options = nn
            bt.value = "Select:"

        if change.new == "Done":
            updateDict("D")

        if change.new == "Start Over":
            sl.options = opts
            updateDict("S")
            bt.value = "Select:"


# ----------------------------------------------------------------------------
# relabel
# ----------------------------------------------------------------------------
def relabel(labelArray: numpy.ndarray, lookup: dict) -> numpy.ndarray:
    newLab = labelArray.copy()

    for k, v in lookup.items():
        if len(v) == 1 and k == v[0]:
            continue
        newLab = numpy.where(numpy.isin(newLab, v), k, newLab)

    return newLab


# ----------------------------------------------------------------------------
# updateDict
# ----------------------------------------------------------------------------
def updateDict(op: str) -> None:
    if op == "N":
        key = list(sl.value)[0]
        table[key] = list(sl.value)

    if op == "D":
        if len(sl.options) > 0:
            key = list(sl.options)[0]
            table[key] = list(sl.options)

        print("Final Groups : ", table)

    if op == "S":
        table.clear()


# ----------------------------------------------------------------------------
# updateList
# ----------------------------------------------------------------------------
def updateList(old: list, out: list) -> list:
    return [ele for ele in old if ele not in out]


# Step 1: Clip the input raster

In [4]:
# This is intentionally performed BEFORE ImageHelper ingestion or clustering.
# The full input TIFF is never passed to Clusterer.getClusters().
crop_center(
    src_path=inFile,
    dst_path=clippedInputFile,
    size=clipSize,
)

print(f"Full input:    {inFile}")
print(f"Clipped input: {clippedInputFile}")

Full input:    /explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites/M1126986303LE.ech.cog.tif
Clipped input: M1126986303LE.ech.cog-clip-512.tif


# Step 2: Ingest ONLY the clipped raster

In [5]:
inHelper = ImageHelper()
inHelper.initFromFile(
    inputFile=clippedInputFile,
    noDataValue=noDataValue,
)

print(f"Clustering input shape: {inHelper.getBand().shape}")

Clustering input shape: (512, 512)


# Step 3: Generate first-pass clusters on the clipped raster

In [6]:
# Add singleton dimension for the single input band: (H, W) -> (1, H, W).
# For clipSize=512, clustering operates on only 512x512 pixels.
labels = Clusterer.getClusters(
    bands=numpy.expand_dims(inHelper.getBand(), axis=0),
    numClusters=numClusters,
)

# Because inHelper was created from clippedInputFile, the label GeoTIFF is
# automatically written with the same clipped extent/transform/CRS.
labelsDs = Clusterer.labelsToGeotiff(
    inHelper._dataset,
    labelsFile,
    labels,
)

lHelper = ImageHelper()
lHelper.initFromDataset(labelsDs, noDataValue)

print(f"Clipped labels: {labelsFile}")

Clipped labels: M1126986303LE.ech.cog-clip-512-labels.tif


# Step 4: Display clipped image + clipped labels

In [7]:
# Use localtileserver directly for raster serving. This path works with the
# Jupyter/VS Code loopback bridge and avoids leafmap.add_raster().
image_client = TileClient(str(clippedInputFile), debug=True)
labels_client = TileClient(str(labelsFile), debug=True)

image_layer = get_leaflet_tile_layer(
    image_client,
    vmin=inHelper._minValue,
    vmax=inHelper._maxValue,
    nodata=inHelper._noDataValue,
    opacity=1.0,
)
image_layer.name = clippedInputFile.name

labels_layer = get_leaflet_tile_layer(
    labels_client,
    vmin=lHelper._minValue,
    vmax=lHelper._maxValue,
    nodata=lHelper._noDataValue,
    opacity=0.5,
    colormap="viridis",
)
labels_layer.name = labelsFile.name

print("Image bounds:", image_layer.bounds)
print("Labels bounds:", labels_layer.bounds)

# TEMPORARY TEST: don't let Leaflet restrict tile loading by bounds
image_layer.bounds = None
labels_layer.bounds = None

m = leafmap.Map(
    fullscreen_control=False,
    layers_control=True,
    search_control=False,
    draw_control=False,
    measure_control=False,
    scale_control=False,
    toolbar_control=True,
    center=image_client.center(),
    zoom=image_client.default_zoom,
)

# Remove the default Earth/OpenStreetMap basemap.
m.remove(m.layers[0])

m.add(image_layer)
m.add(labels_layer)

# bounds = image_client.metadata["bounds"]
# m.fit_bounds(
#     [
#         [bounds["bottom"], bounds["left"]],
#         [bounds["top"], bounds["right"]],
#     ]
# )

m.layout.height = "600px"

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import ipywidgets as widgets
from ipyleaflet import WidgetControl

# Unique cluster IDs
cluster_ids = sorted(int(i) for i in numpy.unique(labels))
vmin = min(cluster_ids)
vmax = max(cluster_ids)

cmap = cm.get_cmap("viridis")

rows = []
for cid in cluster_ids:
    # Match the same continuous viridis mapping used by the layer
    if vmax == vmin:
        t = 0.5
    else:
        t = (cid - vmin) / (vmax - vmin)

    hex_color = mcolors.to_hex(cmap(t))

    rows.append(
        f"""
        <div style="display:flex; align-items:center; margin:2px 0;">
            <div style="
                width:18px;
                height:12px;
                background:{hex_color};
                border:1px solid #444;
                margin-right:8px;
                flex:0 0 auto;
            "></div>
            <div style="font-size:12px;">Cluster {cid}</div>
        </div>
        """
    )

legend_html = widgets.HTML(
    value=f"""
    <div style="
        background:white;
        color: #111;
        padding:8px 10px;
        border:1px solid #777;
        border-radius:4px;
        max-height:300px;
        min-width:140px;
        overflow-y:auto;
        box-shadow:0 1px 4px rgba(0,0,0,0.25);
    ">
        <div style="font-weight:bold; margin-bottom:6px;">
            Cluster legend
        </div>
        {''.join(rows)}
    </div>
    """
)

legend_control = WidgetControl(widget=legend_html, position="topright")
m.add(legend_control)
display(m)

INFO:     Started server process [194704]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:37047 (Press CTRL+C to quit)


INFO:     127.0.0.1:39306 - "GET /api/metadata?&filename=%2Fpanfs%2Fccds02%2Fnobackup%2Fpeople%2Fajkerr1%2FLunar_FM%2Ffull_model_lfm%2Flfm%2Fnotebooks%2FM1126986303LE.ech.cog-clip-512.tif HTTP/1.1" 200 OK


INFO:     127.0.0.1:39310 - "GET /api/metadata?&filename=%2Fpanfs%2Fccds02%2Fnobackup%2Fpeople%2Fajkerr1%2FLunar_FM%2Ffull_model_lfm%2Flfm%2Fnotebooks%2FM1126986303LE.ech.cog-clip-512-labels.tif HTTP/1.1" 200 OK
Image bounds: ((0.781832, 23.369889), (0.798735, 23.386794))
Labels bounds: ((0.781832, 23.369889), (0.798735, 23.386794))


Map(center=[0.7902834999999999, 23.378341499999998], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Update the labels
Select multiple values by clicking the mouse or using the arrow keys while pressing Shift, Control, or Command.

In [8]:
opts = list(numpy.unique(labels))

sl = ipywidgets.SelectMultiple(
    options=opts,
    layout=ipywidgets.Layout(height="200px", width="150px"),
)

bt = ipywidgets.ToggleButtons(
    options=["Select:", "Next", "Done", "Start Over"],
    value="Select:",
)

output = ipywidgets.Output()
display(sl, bt, output)
table = {}
bt.observe(handleClick, names="value")

SelectMultiple(layout=Layout(height='200px', width='150px'), options=(np.int32(0), np.int32(1), np.int32(2), n…

ToggleButtons(options=('Select:', 'Next', 'Done', 'Start Over'), value='Select:')

Output()

## Edit the groups
Edit cluster IDs in each group. When finished, proceed to the next cell.

In [10]:
strTab = {}

for item in table:
    strTab[item] = ", ".join(str(i) for i in table[item])

df = pandas.DataFrame(strTab.items(), columns=["Class", "Cluster ID"])
sheet = ipysheet.from_dataframe(df)
sheet.column_width = [1, 5]
sheet

Sheet(cells=(Cell(column_end=0, column_start=0, numeric_format='0[.]0', row_end=1, row_start=0, squeeze_row=Fa…

In [11]:
editedDf = ipysheet.to_dataframe(sheet)
strClusters = editedDf.to_dict()["Cluster ID"]

finalClusters = {}

for key in strClusters:
    strCluster = strClusters[key]
    finalClusters[int(key)] = [
        int(i.strip()) for i in strCluster.split(",") if i.strip()
    ]

print(finalClusters)
newClusters = relabel(labels, finalClusters)

{0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], 1: [16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]}


## Review the updated map

In [12]:
# This is also clipped because inHelper._dataset is the 512x512 input clip.
cmDataset = Clusterer.labelsToGeotiff(
    inHelper._dataset,
    clusterMapFile,
    newClusters,
)

cmHelper = ImageHelper()
cmHelper.initFromDataset(cmDataset, noDataValue)

cluster_client = TileClient(str(clusterMapFile), debug=True)
cluster_layer = get_leaflet_tile_layer(
    cluster_client,
    vmin=cmHelper._minValue,
    vmax=cmHelper._maxValue,
    nodata=cmHelper._noDataValue,
    opacity=0.5,
    colormap="viridis",
)
cluster_layer.name = clusterMapFile.name

# Remove the old 30-cluster legend, if it is still on the map
try:
    m.remove(legend_control)
except Exception:
    pass

# Final grouped class IDs
class_ids = sorted(int(x) for x in numpy.unique(newClusters))

vmin = min(class_ids)
vmax = max(class_ids)

cmap = cm.get_cmap("viridis")

rows = []

for class_id in class_ids:
    # Match the same vmin/vmax normalization used by localtileserver
    if vmax == vmin:
        t = 0.5
    else:
        t = (class_id - vmin) / (vmax - vmin)

    hex_color = mcolors.to_hex(cmap(t))

    rows.append(
        f"""
        <div style="
            display:flex;
            align-items:center;
            margin:3px 0;
            color:#111;
        ">
            <div style="
                width:18px;
                height:14px;
                background:{hex_color};
                border:1px solid #444;
                margin-right:8px;
                flex:0 0 auto;
            "></div>

            <div style="
                font-size:12px;
                color:#111;
                white-space:nowrap;
            ">
                Class {class_id}
            </div>
        </div>
        """
    )

final_legend_html = widgets.HTML(
    value=f"""
    <div style="
        background:white;
        color:#111;
        padding:8px 10px;
        border:1px solid #777;
        border-radius:4px;
        min-width:120px;
        box-shadow:0 1px 4px rgba(0,0,0,0.25);
    ">
        <div style="
            font-weight:bold;
            margin-bottom:6px;
            color:#111;
        ">
            Final classes
        </div>

        {''.join(rows)}
    </div>
    """
)

final_legend_control = WidgetControl(
    widget=final_legend_html,
    position="topright",
)

m.add(final_legend_control)

# Remove original first-pass labels
if labels_layer in m.layers:
    m.remove(labels_layer)

# Remove an older final layer if this cell is being rerun
for layer in list(m.layers):
    if layer.name == clusterMapFile.name:
        m.remove(layer)

# Add the newly generated final labels
m.add(cluster_layer)

display(m)

INFO:     127.0.0.1:37594 - "GET /api/metadata?&filename=%2Fpanfs%2Fccds02%2Fnobackup%2Fpeople%2Fajkerr1%2FLunar_FM%2Ffull_model_lfm%2Flfm%2Fnotebooks%2FM1126986303LE.ech.cog-clip-512-cluster-map.tif HTTP/1.1" 200 OK


Map(bottom=2088244.0, center=[0.7902834999999999, 23.378341499999998], controls=(ZoomControl(options=['positio…